In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
BASE_DIR = Path(".")

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Load Raw Dataset

In [ ]:
df_estate = pd.read_csv(
    RAW_DIR / "estate_master.csv"
)

df_weather = pd.read_csv(
    RAW_DIR / "weather_data.csv"
)

df_operation = pd.read_csv(
    RAW_DIR / "operation_data.csv"
)

df_production = pd.read_csv(
    RAW_DIR / "production_data.csv"
)

In [ ]:
# convert date
df_weather["date"] = pd.to_datetime(
    df_weather["date"]
)

df_operation["date"] = pd.to_datetime(
    df_operation["date"]
)

df_production["date"] = pd.to_datetime(
    df_production["date"]
)

## Initial Dataset Overview

In [ ]:
datasets = {
    "Estate": df_estate,
    "Weather": df_weather,
    "Operation": df_operation,
    "Production": df_production
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])


Estate
----------------------------------------
Rows: 10
Columns: 5

Weather
----------------------------------------
Rows: 845
Columns: 4

Operation
----------------------------------------
Rows: 845
Columns: 5

Production
----------------------------------------
Rows: 845
Columns: 5


## Data Type Validation

In [ ]:
df_estate.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   estate_id      10 non-null     object
 1   estate_name    10 non-null     object
 2   region         10 non-null     object
 3   area_ha        10 non-null     int64 
 4   planting_year  10 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 532.0+ bytes


In [ ]:
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         845 non-null    datetime64[ns]
 1   estate_id    845 non-null    object        
 2   rainfall_mm  830 non-null    float64       
 3   rainy_days   845 non-null    int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 26.5+ KB


In [ ]:
df_operation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             845 non-null    datetime64[ns]
 1   estate_id        845 non-null    object        
 2   fertilizer_kg    835 non-null    float64       
 3   working_days     845 non-null    int64         
 4   disruption_days  845 non-null    int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 33.1+ KB


In [ ]:
df_production.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   845 non-null    datetime64[ns]
 1   estate_id              845 non-null    object        
 2   tbs_production_ton     837 non-null    float64       
 3   cpo_production_ton     845 non-null    float64       
 4   kernel_production_ton  845 non-null    float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 33.1+ KB


## Missing Value Assessment

In [ ]:
def missing_summary(df):
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        ).round(2)
    })

    return result[
        result["missing_count"] > 0
    ].sort_values(
        "missing_count",
        ascending=False
    )

print("WEATHER")
display(missing_summary(df_weather))

print("OPERATION")
display(missing_summary(df_operation))

print("PRODUCTION")
display(missing_summary(df_production))

WEATHER


,missing_count,missing_percentage
rainfall_mm,15,1.78


OPERATION


,missing_count,missing_percentage
fertilizer_kg,10,1.18


PRODUCTION


,missing_count,missing_percentage
tbs_production_ton,8,0.95


## Duplicate Validation

In [ ]:
# weather
weather_duplicates = df_weather[
    df_weather.duplicated(
        subset=["estate_id", "date"],
        keep=False
    )
].sort_values(
    ["estate_id", "date"]
)

weather_duplicates

,date,estate_id,rainfall_mm,rainy_days
30,2021-07-01,EST001,161.00,10
842,2021-07-01,EST001,161.00,10
96,2020-01-01,EST002,260.40,16
844,2020-01-01,EST002,260.40,16
599,2019-12-01,EST008,304.40,21
843,2019-12-01,EST008,304.40,21
695,2020-12-01,EST009,216.50,12
840,2020-12-01,EST009,216.50,12
816,2024-01-01,EST010,225.20,15
841,2024-01-01,EST010,225.20,15


In [ ]:
# operation
operation_duplicates = df_operation[
    df_operation.duplicated(
        subset=["estate_id", "date"],
        keep=False
    )
].sort_values(
    ["estate_id", "date"]
)

operation_duplicates

,date,estate_id,fertilizer_kg,working_days,disruption_days
30,2021-07-01,EST001,"35,946.00",23,0
842,2021-07-01,EST001,"35,946.00",23,0
96,2020-01-01,EST002,"40,232.00",23,3
844,2020-01-01,EST002,"40,232.00",23,3
599,2019-12-01,EST008,"41,088.00",26,1
843,2019-12-01,EST008,"41,088.00",26,1
695,2020-12-01,EST009,"33,198.00",24,1
840,2020-12-01,EST009,"33,198.00",24,1
816,2024-01-01,EST010,"39,945.00",25,0
841,2024-01-01,EST010,"39,945.00",25,0


In [ ]:
# production
production_duplicates = df_production[
    df_production.duplicated(
        subset=["estate_id", "date"],
        keep=False
    )
].sort_values(
    ["estate_id", "date"]
)

production_duplicates

,date,estate_id,tbs_production_ton,cpo_production_ton,kernel_production_ton
30,2021-07-01,EST001,"8,785.10","1,569.00",380.10
842,2021-07-01,EST001,"8,785.10","1,569.00",380.10
96,2020-01-01,EST002,"7,041.80","1,346.30",337.00
844,2020-01-01,EST002,"7,041.80","1,346.30",337.00
599,2019-12-01,EST008,"5,437.40","1,121.90",293.30
843,2019-12-01,EST008,"5,437.40","1,121.90",293.30
695,2020-12-01,EST009,"5,974.70","1,121.10",289.40
840,2020-12-01,EST009,"5,974.70","1,121.10",289.40
816,2024-01-01,EST010,"8,985.60","2,065.60",510.90
841,2024-01-01,EST010,"8,985.60","2,065.60",510.90


## Remove Duplicate Records

In [ ]:
df_weather = df_weather.drop_duplicates(
    subset=["estate_id", "date"],
    keep="first"
).copy()

df_operation = df_operation.drop_duplicates(
    subset=["estate_id", "date"],
    keep="first"
).copy()

df_production = df_production.drop_duplicates(
    subset=["estate_id", "date"],
    keep="first"
).copy()

In [ ]:
print(
    "Weather duplicates:",
    df_weather.duplicated(
        subset=["estate_id", "date"]
    ).sum()
)

print(
    "Operation duplicates:",
    df_operation.duplicated(
        subset=["estate_id", "date"]
    ).sum()
)

print(
    "Production duplicates:",
    df_production.duplicated(
        subset=["estate_id", "date"]
    ).sum()
)

Weather duplicates: 0
Operation duplicates: 0
Production duplicates: 0


## Validate Estate IDs

In [ ]:
valid_estates = set(
    df_estate["estate_id"]
)

print(
    "Weather invalid estate:",
    (~df_weather["estate_id"].isin(valid_estates)).sum()
)

print(
    "Operation invalid estate:",
    (~df_operation["estate_id"].isin(valid_estates)).sum()
)

print(
    "Production invalid estate:",
    (~df_production["estate_id"].isin(valid_estates)).sum()
)

Weather invalid estate: 0
Operation invalid estate: 0
Production invalid estate: 0


## Validate Date Range

In [ ]:
START_DATE = pd.Timestamp("2019-01-01")
END_DATE = pd.Timestamp("2025-12-01")

In [ ]:
def check_date_range(df, date_column="date"):
    invalid = df[
        (df[date_column] < START_DATE) |
        (df[date_column] > END_DATE)
    ]

    return invalid

In [ ]:
print(
    "Weather invalid dates:",
    len(check_date_range(df_weather))
)

print(
    "Operation invalid dates:",
    len(check_date_range(df_operation))
)

print(
    "Production invalid dates:",
    len(check_date_range(df_production))
)

Weather invalid dates: 0
Operation invalid dates: 0
Production invalid dates: 0


## Validate Rainfall

In [ ]:
invalid_rainfall = df_weather[
    df_weather["rainfall_mm"] < 0
]

invalid_rainfall

,date,estate_id,rainfall_mm,rainy_days
27,2021-04-01,EST001,-50.00,15
40,2022-05-01,EST001,-50.00,18
347,2019-12-01,EST005,-50.00,15
385,2023-02-01,EST005,-50.00,16
761,2019-06-01,EST010,-50.00,13


In [ ]:
df_weather.loc[
    df_weather["rainfall_mm"] < 0,
    "rainfall_mm"
] = np.nan

## Validate Rainy Days

In [ ]:
invalid_rainy_days = df_weather[
    (df_weather["rainy_days"] < 0) |
    (df_weather["rainy_days"] > 31)
]

invalid_rainy_days

,date,estate_id,rainfall_mm,rainy_days


In [ ]:
df_weather.loc[
    (df_weather["rainy_days"] < 0) |
    (df_weather["rainy_days"] > 31),
    "rainy_days"
] = np.nan

## Validate Fertilizer

In [ ]:
invalid_fertilizer = df_operation[
    df_operation["fertilizer_kg"] < 0
]

invalid_fertilizer

,date,estate_id,fertilizer_kg,working_days,disruption_days
162,2025-07-01,EST002,"-1,000.00",24,1
441,2020-10-01,EST006,"-1,000.00",25,3
555,2023-04-01,EST007,"-1,000.00",25,2
733,2024-02-01,EST009,"-1,000.00",22,2
823,2024-08-01,EST010,"-1,000.00",23,0


In [ ]:
df_operation.loc[
    df_operation["fertilizer_kg"] < 0,
    "fertilizer_kg"
] = np.nan

## Validate Working Days

In [ ]:
invalid_working_days = df_operation[
    (df_operation["working_days"] < 0) |
    (df_operation["working_days"] > 31)
]

invalid_working_days

,date,estate_id,fertilizer_kg,working_days,disruption_days
254,2019-03-01,EST004,"39,046.00",45,2
288,2022-01-01,EST004,"45,067.00",45,2
348,2020-01-01,EST005,"59,282.00",45,1
424,2019-05-01,EST006,"36,654.00",45,0
794,2022-03-01,EST010,"38,898.00",45,1


In [ ]:
df_operation.loc[
    (df_operation["working_days"] < 0) |
    (df_operation["working_days"] > 31),
    "working_days"
] = np.nan

## Validate Disruption Days

In [ ]:
invalid_disruption = df_operation[
    (df_operation["disruption_days"] < 0) |
    (df_operation["disruption_days"] > 31)
]

invalid_disruption

,date,estate_id,fertilizer_kg,working_days,disruption_days


In [ ]:
df_operation.loc[
    (df_operation["disruption_days"] < 0) |
    (df_operation["disruption_days"] > 31),
    "disruption_days"
] = np.nan

## Validate Production

In [ ]:
production_columns = [
    "tbs_production_ton",
    "cpo_production_ton",
    "kernel_production_ton"
]

for col in production_columns:
    invalid = df_production[
        df_production[col] < 0
    ]

    print(
        f"{col}: {len(invalid)} invalid records"
    )

tbs_production_ton: 5 invalid records
cpo_production_ton: 0 invalid records
kernel_production_ton: 0 invalid records


In [ ]:
for col in production_columns:
    df_production.loc[
        df_production[col] < 0,
        col
    ] = np.nan

## Missing Value Treatment

In [ ]:
df_weather = df_weather.sort_values(
    ["estate_id", "date"]
)

df_operation = df_operation.sort_values(
    ["estate_id", "date"]
)

df_production = df_production.sort_values(
    ["estate_id", "date"]
)

In [ ]:
# Interpolate Weather
weather_numeric = [
    "rainfall_mm",
    "rainy_days"
]

for col in weather_numeric:
    df_weather[col] = (
        df_weather
        .groupby("estate_id")[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

In [ ]:
# Interpolate Operational Data
operation_numeric = [
    "fertilizer_kg",
    "working_days",
    "disruption_days"
]

for col in operation_numeric:
    df_operation[col] = (
        df_operation
        .groupby("estate_id")[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

## Production Missing Value

In [ ]:
for col in production_columns:
    df_production[col] = (
        df_production
        .groupby("estate_id")[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

## Validasi Missing Setelah Cleaning

In [ ]:
print("Weather missing:")
print(df_weather.isna().sum())

print("\nOperation missing:")
print(df_operation.isna().sum())

print("\nProduction missing:")
print(df_production.isna().sum())

Weather missing:
date           0
estate_id      0
rainfall_mm    0
rainy_days     0
dtype: int64

Operation missing:
date               0
estate_id          0
fertilizer_kg      0
working_days       0
disruption_days    0
dtype: int64

Production missing:
date                     0
estate_id                0
tbs_production_ton       0
cpo_production_ton       0
kernel_production_ton    0
dtype: int64


## Outlier Detection

In [ ]:
def detect_iqr_outliers(
    df,
    column,
    group_column="estate_id"
):
    result = []

    for group, data in df.groupby(group_column):

        q1 = data[column].quantile(0.25)
        q3 = data[column].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (
            (data[column] < lower) |
            (data[column] > upper)
        )

        temp = data.loc[mask].copy()
        temp["lower_bound"] = lower
        temp["upper_bound"] = upper

        result.append(temp)

    if result:
        return pd.concat(result)

    return pd.DataFrame()

## Detect TBS Outliers

In [ ]:
tbs_outliers = detect_iqr_outliers(
    df_production,
    "tbs_production_ton"
)

print(
    "Detected TBS outliers:",
    len(tbs_outliers)
)

Detected TBS outliers: 8


In [ ]:
tbs_outliers[
    [
        "estate_id",
        "date",
        "tbs_production_ton",
        "lower_bound",
        "upper_bound"
    ]
]

,estate_id,date,tbs_production_ton,lower_bound,upper_bound
92,EST002,2019-09-01,"18,902.50","5,407.05","10,636.85"
236,EST003,2024-09-01,"23,112.20","7,950.47","13,819.88"
306,EST004,2023-07-01,"32,037.20","7,314.28","13,364.88"
377,EST005,2022-06-01,"15,688.20","9,113.97","15,341.38"
487,EST006,2024-08-01,"23,273.50","5,223.40","12,455.40"
521,EST007,2020-06-01,"25,373.20","6,327.73","14,500.33"
599,EST008,2019-12-01,"5,437.40","5,604.90","11,090.10"
665,EST008,2025-06-01,"11,573.30","5,604.90","11,090.10"


In [ ]:
# Jangan Langsung Hapus Outlier
# I first investigate whether the outlier represents a genuine operational event or a data quality issue.
# If it is a valid observation, I retain it; if it is caused by an error, I correct or treat it appropriately.

In [ ]:
tbs_median = (
    df_production
    .groupby("estate_id")["tbs_production_ton"]
    .transform("median")
)

# tandai dulu
df_production["tbs_outlier_flag"] = False

outlier_index = tbs_outliers.index

df_production.loc[
    outlier_index,
    "tbs_outlier_flag"
] = True

## Treatment Outlier

In [ ]:
df_production.loc[
    df_production["tbs_outlier_flag"],
    "tbs_production_ton"
] = np.nan

In [ ]:
df_production["tbs_production_ton"] = (
    df_production
    .groupby("estate_id")["tbs_production_ton"]
    .transform(
        lambda x: x.interpolate(
            method="linear",
            limit_direction="both"
        )
    )
)

## Cross-table Validation

In [ ]:
# memastikan semua tabel punya kombinasi: estate_id + date

weather_keys = set(
    zip(
        df_weather["estate_id"],
        df_weather["date"]
    )
)

operation_keys = set(
    zip(
        df_operation["estate_id"],
        df_operation["date"]
    )
)

production_keys = set(
    zip(
        df_production["estate_id"],
        df_production["date"]
    )
)

In [ ]:
print(
    "Weather vs Operation:",
    weather_keys == operation_keys
)

print(
    "Weather vs Production:",
    weather_keys == production_keys
)

Weather vs Operation: True
Weather vs Production: True


## Validate Production Relationship

In [ ]:
# CPO tidak boleh lebih besar daripada TBS.
invalid_cpo = df_production[
    df_production["cpo_production_ton"]
    > df_production["tbs_production_ton"]
]

print(
    "Invalid CPO records:",
    len(invalid_cpo)
)

invalid_kernel = df_production[
    df_production["kernel_production_ton"]
    > df_production["tbs_production_ton"]
]

print(
    "Invalid Kernel records:",
    len(invalid_kernel)
)

Invalid CPO records: 0
Invalid Kernel records: 0


## Validate Extraction Rate

In [ ]:
df_production["cpo_extraction_rate"] = (
    df_production["cpo_production_ton"]
    / df_production["tbs_production_ton"]
)

df_production["kernel_extraction_rate"] = (
    df_production["kernel_production_ton"]
    / df_production["tbs_production_ton"]
)

In [ ]:
invalid_oer = df_production[
    (df_production["cpo_extraction_rate"] < 0.15) |
    (df_production["cpo_extraction_rate"] > 0.25)
]

print(
    "Invalid CPO extraction rate:",
    len(invalid_oer)
)

Invalid CPO extraction rate: 0


In [ ]:
invalid_ker = df_production[
    (df_production["kernel_extraction_rate"] < 0.03) |
    (df_production["kernel_extraction_rate"] > 0.07)
]

print(
    "Invalid kernel extraction rate:",
    len(invalid_ker)
)

Invalid kernel extraction rate: 0


## Final Data Quality Report

In [ ]:
quality_report = pd.DataFrame({
    "Dataset": [
        "Estate Master",
        "Weather",
        "Operation",
        "Production"
    ],

    "Rows": [
        len(df_estate),
        len(df_weather),
        len(df_operation),
        len(df_production)
    ],

    "Columns": [
        df_estate.shape[1],
        df_weather.shape[1],
        df_operation.shape[1],
        df_production.shape[1]
    ],

    "Missing Values": [
        df_estate.isna().sum().sum(),
        df_weather.isna().sum().sum(),
        df_operation.isna().sum().sum(),
        df_production.isna().sum().sum()
    ],

    "Duplicate Keys": [
        0,
        df_weather.duplicated(
            ["estate_id", "date"]
        ).sum(),
        df_operation.duplicated(
            ["estate_id", "date"]
        ).sum(),
        df_production.duplicated(
            ["estate_id", "date"]
        ).sum()
    ]
})

quality_report

,Dataset,Rows,Columns,Missing Values,Duplicate Keys
0,Estate Master,10,5,0,0
1,Weather,840,4,0,0
2,Operation,840,5,0,0
3,Production,840,8,0,0


## Save Processed Dataset

In [ ]:
df_estate.to_csv(
    PROCESSED_DIR / "estate_master_cleaned.csv",
    index=False
)

df_weather.to_csv(
    PROCESSED_DIR / "weather_data_cleaned.csv",
    index=False
)

df_operation.to_csv(
    PROCESSED_DIR / "operation_data_cleaned.csv",
    index=False
)

df_production.to_csv(
    PROCESSED_DIR / "production_data_cleaned.csv",
    index=False
)

# empat jenis validasi:
1. Structural validation
- datatype
- duplicate key
- date range
- estate ID
2. Completeness validation
- missing values
3. Range validation
- rainfall ≥ 0
- fertilizer ≥ 0
- working days ≤ 31
- production ≥ 0
4. Business-rule validation
- CPO ≤ TBS
- kernel ≤ TBS
- extraction rate dalam reasonable range
- estate-month harus konsisten antar tabel

#### Data Quality & Validation: Performed structural, completeness, range, and business-rule validation across production, weather, and operational datasets; handled missing values, duplicate records, invalid observations, and production outliers before downstream modeling.